In [34]:
import pandas as pd

In [35]:
# Load the dataset
df = pd.read_csv('Data/coffee_reviews_parsed.csv')

# Inspect the dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9009 entries, 0 to 9008
Data columns (total 23 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   URL                  9009 non-null   object 
 1   all_text             9009 non-null   object 
 2   Rating               9003 non-null   float64
 3   Roaster              9003 non-null   object 
 4   Coffee Name          9003 non-null   object 
 5   Roaster Location     8894 non-null   object 
 6   Coffee Origin        8436 non-null   object 
 7   Roast Level          8493 non-null   object 
 8   Agtron               8898 non-null   object 
 9   Est. Price           6859 non-null   object 
 10  Review Date          9008 non-null   object 
 11  Aroma                8828 non-null   float64
 12  Acidity              3709 non-null   float64
 13  Acidity/Structure    3809 non-null   float64
 14  Body                 8885 non-null   float64
 15  Flavor               8881 non-null   f

In [36]:
# Convert 'Review Date' to datetime and find the oldest and newest review dates
df['Review Date'] = pd.to_datetime(df['Review Date'].astype(str).str.strip(), format='%B %Y', errors='coerce')
oldest_review = df['Review Date'].min()
newest_review = df['Review Date'].max()

# Print the oldest and newest review dates, as well as the columns and the first few rows of the DataFrame
print(f'Oldest review date: {oldest_review.date()}')
print(f'Newest review date: {newest_review.date()}')

Oldest review date: 1997-02-01
Newest review date: 2026-03-01


In [37]:
# Check for duplicate URLs
print(f"Number of duplicate URLs: {df.duplicated(subset=['URL']).sum()}\n")

# Check for missing values
print("Missing values in each column:")
print(df.isna().sum())

Number of duplicate URLs: 0

Missing values in each column:
URL                       0
all_text                  0
Rating                    6
Roaster                   6
Coffee Name               6
Roaster Location        115
Coffee Origin           573
Roast Level             516
Agtron                  111
Est. Price             2150
Review Date               1
Aroma                   181
Acidity                5300
Acidity/Structure      5200
Body                    124
Flavor                  128
Aftertaste              982
With Milk              7790
Blind Assessment         12
Notes                     4
Who Should Drink It    4982
Bottom Line            4191
Combined_Acidity       1491
dtype: int64


# Normalize Blind Assessment

In [38]:
# Normalize the 'Blind Assessment' text data

print("Before normalization:")
print(df["Blind Assessment"].head())

df["Blind Assessment"] = (
    df["Blind Assessment"]
    .str.strip() # Remove leading and trailing whitespace
    .str.lower() # Convert to lowercase
    .str.replace(r"\s+", " ", regex=True)  # Replace multiple spaces with a single space
)

print("\nAfter normalization:")
print(df["Blind Assessment"].head())

Before normalization:
0    Evaluated as espresso. Smoothly round aroma: t...
1    Produced from an ESE pod on a FrancisFrancis! ...
2    Bittersweet but balanced; chocolaty. Dark choc...
3    Evaluated at proportions of 5 grams of instant...
4    In the aroma caramel and wet burned wood notes...
Name: Blind Assessment, dtype: object

After normalization:
0    evaluated as espresso. smoothly round aroma: t...
1    produced from an ese pod on a francisfrancis! ...
2    bittersweet but balanced; chocolaty. dark choc...
3    evaluated at proportions of 5 grams of instant...
4    in the aroma caramel and wet burned wood notes...
Name: Blind Assessment, dtype: object


# Remove duplicates, empty, and blind assessment with less than 100 characters.

In [39]:
before = len(df)

# Filter out rows where 'Blind Assessment' is empty or has fewer than 100 characters
s = df["Blind Assessment"].fillna("").astype(str).str.strip()
df = df[s.str.len() >= 100].copy()

print("Rows before:", before)
print("Rows after:", len(df))
print("Rows removed:", before - len(df))

Rows before: 9009
Rows after: 8899
Rows removed: 110


In [40]:
# Check for duplicates in 'Blind Assessment' and remove them
n_dupes = df.duplicated(subset=["Blind Assessment"]).sum()
print("Exact duplicate rows (excluding first occurrence):", n_dupes)

dupes_df = df[df.duplicated(subset=["Blind Assessment"], keep=False)] \
            .sort_values("Blind Assessment")

print("\nDuplicate rows:")
print(dupes_df[["Blind Assessment"]].head(30))

df = df.drop_duplicates(subset=["Blind Assessment"], keep="first")
print("\nDataFrame shape after removing duplicates:", df.shape)

Exact duplicate rows (excluding first occurrence): 10

Duplicate rows:
                                       Blind Assessment
5727  a high score but a relatively low wow quotient...
5395  a high score but a relatively low wow quotient...
8733  a light, bright, fragrantly smooth cup. when h...
8732  a light, bright, fragrantly smooth cup. when h...
5396  bright but well-matrixed acidity, full body, a...
5750  bright but well-matrixed acidity, full body, a...
1653  deeply rich, chocolaty and fruit-toned. raspbe...
1654  deeply rich, chocolaty and fruit-toned. raspbe...
8843  evaluated as a green coffee sample-roasted to ...
8844  evaluated as a green coffee sample-roasted to ...
924   reader peter lynagh nominated this coffee, rat...
923   reader peter lynagh nominated this coffee, rat...
5040  richly chocolaty, deeply sweet. dark chocolate...
2340  richly chocolaty, deeply sweet. dark chocolate...
6222  sweetly and richly fermenty: rye whisky, lilac...
6661  sweetly and richly fermenty

# Save to csv

In [41]:
# Save the cleaned DataFrame to a new CSV file
df.to_csv('Data/final_coffee_reviews.csv', index=False)

df_test = pd.read_csv('Data/final_coffee_reviews.csv')
print(df_test.shape)

(8889, 23)
